# 05 — Viability and finance

This notebook asks:

> Are candidates with more campaign-finance activity more likely to clear the
> observed mention-based viability threshold?

Primary variable: **total fundraising amount**.

We also inspect contribution-record count and, if the spending notebook has
already been run, total reported spending.

Viability comes from the VoteKit support table created in notebook 03:

`is_viable = mentions >= district STV threshold`

This is descriptive, not causal.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

YEAR = 2024

cwd = Path.cwd()

if (cwd / "pyproject.toml").exists():
    ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not find repository root.")

SUPPORT_PATH = (
    ROOT / "data" / "processed" / "ballot_support" / str(YEAR)
    / "candidate_ballot_support_2024.csv"
)

FUNDRAISING_PATH = (
    ROOT / "data" / "processed" / "fundraising" / str(YEAR)
    / "candidate_fundraising_summary.csv"
)

SPENDING_PATH = (
    ROOT / "data" / "processed" / "spending" / str(YEAR)
    / "candidate_spending_summary.csv"
)

OUTPUT_DIR = (
    ROOT / "data" / "processed" / "viability" / str(YEAR)
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Support exists:", SUPPORT_PATH.exists())
print("Fundraising exists:", FUNDRAISING_PATH.exists())
print("Spending exists:", SPENDING_PATH.exists())

## 1. Merge viability with fundraising

In [ ]:
support = pd.read_csv(SUPPORT_PATH, low_memory=False)
fundraising = pd.read_csv(FUNDRAISING_PATH, low_memory=False)

analysis = support.merge(
    fundraising,
    on=["year", "district", "candidate_key"],
    how="inner",
    suffixes=("_ballot", "_finance"),
    validate="one_to_one",
)

analysis["viable_int"] = (
    analysis["is_viable"].astype(bool).astype(int)
)

print("Candidates in fundraising + viability sample:", len(analysis))
display(analysis["is_viable"].value_counts(dropna=False))

print("\nBy district:")
display(pd.crosstab(analysis["district"], analysis["is_viable"]))

## 2. Compare fundraising totals by viability

In [ ]:
fundraising_by_viability = (
    analysis
    .groupby("is_viable", as_index=False)
    .agg(
        candidates=("candidate_key", "size"),
        mean_total_amount=("total_amount", "mean"),
        median_total_amount=("total_amount", "median"),
        mean_contribution_count=("total_contribution_count", "mean"),
        median_contribution_count=("total_contribution_count", "median"),
    )
)

display(fundraising_by_viability)

## 3. Correlations with binary viability

With a 0/1 viability variable, Pearson correlation with a continuous variable
is the point-biserial correlation.

We report raw and log-transformed fundraising measures.

In [ ]:
analysis["log_total_amount"] = np.log1p(analysis["total_amount"])
analysis["log_contribution_count"] = np.log1p(
    analysis["total_contribution_count"]
)

CORRELATION_VARS = [
    "total_amount",
    "log_total_amount",
    "total_contribution_count",
    "log_contribution_count",
]

rows = []

for variable in CORRELATION_VARS:
    for district in [None] + sorted(analysis["district"].unique().tolist()):
        if district is None:
            pair = analysis[["viable_int", variable]].dropna()
            scope = "all_districts"
        else:
            pair = analysis.loc[
                analysis["district"].eq(district),
                ["viable_int", variable],
            ].dropna()
            scope = "district"

        if len(pair) < 3 or pair["viable_int"].nunique() < 2:
            continue

        rows.append(
            {
                "scope": scope,
                "district": district,
                "finance_variable": variable,
                "n": len(pair),
                "viability_correlation": (
                    pair["viable_int"].corr(pair[variable])
                ),
            }
        )

viability_correlations = pd.DataFrame(rows)
display(viability_correlations)

## 4. Simple fundraising/viability plot

In [ ]:
plot_data = analysis.loc[
    analysis["total_amount"].gt(0)
].copy()

rng = np.random.default_rng(42)
jitter = rng.normal(
    loc=0,
    scale=0.035,
    size=len(plot_data),
)

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    plot_data["total_amount"],
    plot_data["viable_int"] + jitter,
    alpha=0.7,
)

ax.set_xscale("log")
ax.set_yticks([0, 1])
ax.set_yticklabels(["Not viable", "Viable"])
ax.set_xlabel("Total private fundraising ($, log scale)")
ax.set_ylabel("")
ax.set_title("Fundraising and mention-based viability")

plt.tight_layout()
plt.show()

## 5. Optional: total reported spending and viability

If notebook 04 has been run, add spending to the same candidate table.

In [ ]:
if SPENDING_PATH.exists():
    spending = pd.read_csv(SPENDING_PATH, low_memory=False)

    analysis = analysis.merge(
        spending[
            [
                "year",
                "district",
                "candidate_key",
                "total_spending",
                "expenditure_count",
            ]
        ],
        on=["year", "district", "candidate_key"],
        how="left",
        validate="one_to_one",
    )

    analysis["log_total_spending"] = np.log1p(
        analysis["total_spending"]
    )

    spending_pair = analysis[
        ["viable_int", "total_spending", "log_total_spending"]
    ].dropna()

    print(
        "Correlation viability vs total spending:",
        spending_pair["viable_int"].corr(
            spending_pair["total_spending"]
        ),
    )

    print(
        "Correlation viability vs log total spending:",
        spending_pair["viable_int"].corr(
            spending_pair["log_total_spending"]
        ),
    )

    display(
        analysis.groupby("is_viable", as_index=False).agg(
            candidates_with_spending=("total_spending", "count"),
            mean_total_spending=("total_spending", "mean"),
            median_total_spending=("total_spending", "median"),
        )
    )
else:
    print(
        "Spending output not found. Run 04_spending_profiles.ipynb "
        "and rerun this cell."
    )

## 6. District-level summary

In [ ]:
district_summary = (
    analysis
    .groupby(["district", "is_viable"], as_index=False)
    .agg(
        candidates=("candidate_key", "size"),
        mean_total_amount=("total_amount", "mean"),
        median_total_amount=("total_amount", "median"),
        mean_contribution_count=("total_contribution_count", "mean"),
    )
)

display(district_summary)

## 7. Export

In [ ]:
analysis_path = OUTPUT_DIR / "candidate_viability_finance_2024.csv"
correlation_path = OUTPUT_DIR / "viability_finance_correlations_2024.csv"
district_path = OUTPUT_DIR / "viability_finance_by_district_2024.csv"

analysis.to_csv(analysis_path, index=False)
viability_correlations.to_csv(correlation_path, index=False)
district_summary.to_csv(district_path, index=False)

print("SAVED", analysis_path)
print("SAVED", correlation_path)
print("SAVED", district_path)